In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Conv2D, MaxPooling2D, Conv2DTranspose, Dense, Cropping2D
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Input, Dense, Reshape
from tensorflow.keras.models import Model
from tensorflow.keras import layers, models




In [ ]:
# Prepare your image data generator for training

batch_size = 32

path_train = "/path/to/private-data/Datagen_Left/Inlet (5 marking)/train"
path_val = "/path/to/private-data/Datagen_Left/Inlet (5 marking)/validation"


train_datagen = ImageDataGenerator(rescale=1./255)
train_generator = train_datagen.flow_from_directory(
    path_train,
    target_size=(400, 320),
    batch_size=batch_size,
    class_mode='input', color_mode='rgb'  # since we are not interested in labels for autoencoder
)

datagen = ImageDataGenerator(rescale=1./255)
validation_generator = datagen.flow_from_directory(path_val,  batch_size=batch_size, class_mode='input', shuffle=True, color_mode='rgb')

In [ ]:
resnet_encoder = ResNet50(include_top=False, weights='imagenet', input_tensor=Input(shape=input_shape))

# If you choose to freeze the ResNet layers to prevent them from being updated during training
for layer in resnet_encoder.layers:
    layer.trainable = False

# Now, define your custom decoder model
decoder = Sequential([
    Conv2DTranspose(32, (3, 3), strides=(2, 2), activation='relu', padding='same'),
    Conv2DTranspose(64, (3, 3), strides=(2, 2), activation='relu', padding='same'),
    Conv2DTranspose(128, (3, 3), strides=(2, 2), activation='relu', padding='same'),
    Conv2DTranspose(32, (3, 3), strides=(2, 2), activation='relu', padding='same'),
    Conv2DTranspose(64, (3, 3), strides=(2, 2), activation='relu', padding='same'),
    Conv2DTranspose(128, (3, 3), strides=(2, 2), activation='relu', padding='same'),
    Conv2DTranspose(64, (3, 3), strides=(2, 2), activation='relu', padding='same'),
    Conv2D(3, (3, 3), activation='sigmoid', padding='same'),
    Cropping2D(cropping=((56, 56), (32, 32)))
])

# Create a new model combining ResNet50 encoder and your decoder
model = Model(inputs=resnet_encoder.input, outputs=decoder(resnet_encoder.output))

# Compile the model
model.compile(loss='mean_squared_error', optimizer='adam', metrics=['accuracy'])

# Display the model summary
model.summary()

In [ ]:
epoch = 1000
batch_size = 25
steps_per_epoch =len(train_generator)
autoencoder.fit(train_generator, validation_data= validation_generator, epochs=epoch, batch_size= batch_size, steps_per_epoch=steps_per_epoch)

In [ ]:
autoencoder.save('autoencoder_model.h5')  # Saves the model in HDF5 format


In [ ]:
from tensorflow.keras.models import load_model

# Load the model from disk
loaded_autoencoder = load_model('autoencoder_model.h5')

In [ ]:
# Evaluate the model
evaluation = loaded_autoencoder.evaluate(validation_generator)
print('Model Loss on Validation Data:', evaluation)

# Predict using the model
predictions = loaded_autoencoder.predict(validation_generator)